In [ ]:
# ===============================
# 3D Image Rotation App (Colab Ready)
# - Image Editing: Qwen 3D Camera Control
# - LLM: LLaMA 3.1 (via Groq)
# - UI: Gradio (Interactive sliders for axis rotation)
# ===============================

# Step 1: Install dependencies
!pip install -q transformers accelerate pillow gradio groq diffusers huggingface_hub

# Step 2: Imports
import torch
from PIL import Image
import gradio as gr
from groq import Groq
from diffusers import DiffusionPipeline
import numpy as np

# Step 3: Hugging Face Token (REQUIRED for gated models if any)
HF_TOKEN = "YOUR_HF_TOKEN"  # Replace with your Hugging Face token

# Step 4: Load Image Rotation Model (Qwen 3D Camera)
print("Loading 3D Camera Control Model...")
pipeline = DiffusionPipeline.from_pretrained(
    "multimodalart/qwen-image-multiple-angles-3d-camera",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    use_auth_token=HF_TOKEN
)
pipeline.to("cuda" if torch.cuda.is_available() else "cpu")

# Step 5: Setup Groq (LLaMA 3.1)
GROQ_API_KEY = "YOUR_GROQ_API_KEY"
client = Groq(api_key=GROQ_API_KEY)

# Step 5: LLM Guidance Function

def get_orientation_prompt(user_instruction):
    prompt = f"""
    You are an expert in 3D camera control.
    Convert the user instruction into precise camera orientation (yaw, pitch, roll).
    Instruction: {user_instruction}
    Output format: yaw=..., pitch=..., roll=...
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content

# Step 6: Image Rotation Function

def rotate_image(image, yaw, pitch, roll):
    if image is None:
        return None

    prompt = f"3D rotation with yaw {yaw}, pitch {pitch}, roll {roll}"

    result = pipeline(prompt=prompt, image=image).images[0]
    return result

# Step 7: Combined Function

def process(image, instruction, yaw, pitch, roll):
    if image is None:
        return None, "Upload image first"

    llm_output = get_orientation_prompt(instruction) if instruction else "Manual control used"
    rotated = rotate_image(image, yaw, pitch, roll)

    return rotated, llm_output

# Step 8: Gradio UI

with gr.Blocks(theme=gr.themes.Soft(), title="3D Image Camera Control") as demo:
    gr.Markdown("""
    # 🎥 3D Image Camera Control Studio
    Upload an image → Describe angle OR use sliders → Get rotated 3D view
    Powered by Qwen + LLaMA 3.1 (Groq)
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Upload Image")
            instruction = gr.Textbox(label="Describe Orientation",
                                     placeholder="e.g., rotate left, top-down view")

            yaw = gr.Slider(-180, 180, value=0, step=1, label="Yaw (Left/Right)")
            pitch = gr.Slider(-90, 90, value=0, step=1, label="Pitch (Up/Down)")
            roll = gr.Slider(-180, 180, value=0, step=1, label="Roll (Tilt)")

            btn = gr.Button("Generate 3D View", variant="primary")

        with gr.Column():
            output_image = gr.Image(label="Rotated Image")
            llm_text = gr.Textbox(label="LLM Interpretation")

    btn.click(
        fn=process,
        inputs=[image_input, instruction, yaw, pitch, roll],
        outputs=[output_image, llm_text]
    )

# Step 9: Launch

demo.launch(share=True)

# ===============================
# FEATURES
# ===============================
# ✔ Upload image
# ✔ Ask orientation in natural language
# ✔ LLaMA converts to camera angles
# ✔ Manual slider override
# ✔ Real-time 3D rotation output
# ===============================
